# Next-Basket Recommendation — Instacart + Sets2Sets

Notebook khởi động cho dự án niên luận của bạn. Mục tiêu là dự đoán giỏ hàng kế tiếp từ lịch sử đơn hàng. Mô hình chính là **Sets2Sets** (dùng mã nguồn tác giả), baseline là **Popularity**. Không cần Drive, PostgreSQL, API ứng dụng hay giao diện web.

**Trình tự:** tải bộ file dự án ZIP → cấu hình Kaggle → tải Instacart trực tiếp vào Colab → tạo dữ liệu cho H = 4, 8, 13 → chạy baseline → thử Sets2Sets trên một H trước.

Lưu ý: source Sets2Sets gốc khá cũ (Python 3.6 / PyTorch 1.0.1 theo README); notebook không giả định rằng nó sẽ chạy nguyên trạng trong Colab hiện tại. Nếu lỗi, hãy xem log đầy đủ thay vì vá mò.

## 1. Cài thư viện và nạp source project

Tải file `NextBasket_Instacart_Sets2Sets.zip` được cung cấp kèm notebook, rồi chọn file đó khi cell dưới yêu cầu. ZIP gồm các module tiền xử lý/metric để không phải chép code giữa nhiều cell.

In [13]:
%pip -q install pandas numpy matplotlib kaggle

from google.colab import files
from pathlib import Path
import zipfile, os, sys, shutil, subprocess, json

# Upload ZIP của project đã tải về từ cuộc trò chuyện này.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise FileNotFoundError('Hãy upload file NextBasket_Instacart_Sets2Sets.zip trước.')
project_zip = Path('/content') / zip_names[0]
with zipfile.ZipFile(project_zip, 'r') as zf:
    zf.extractall('/content')

PROJECT_DIR = Path('/content/NextBasket_Instacart_Sets2Sets')
if not (PROJECT_DIR / 'src' / 'prepare_instacart.py').exists():
    # fallback: find extracted project folder by filename
    matches = list(Path('/content').glob('**/src/prepare_instacart.py'))
    if not matches:
        raise FileNotFoundError('Không tìm thấy src/prepare_instacart.py trong ZIP.')
    PROJECT_DIR = matches[0].parent.parent

sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('Project:', PROJECT_DIR)
print('Source files:', sorted(p.name for p in (PROJECT_DIR / 'src').glob('*.py')))

Saving NextBasket_Instacart_Sets2Sets_Project.zip to NextBasket_Instacart_Sets2Sets_Project (1).zip
Project: /content/NextBasket_Instacart_Sets2Sets
Source files: ['__init__.py', 'evaluate_popularity.py', 'metrics.py', 'prepare_instacart.py', 'run_sets2sets.py']


## 2. Cấu hình Kaggle API

Tạo API Token tại [Kaggle Settings](https://www.kaggle.com/settings) → API. Tải `kaggle.json` về máy và upload khi được hỏi. **Không chia sẻ token hoặc commit lên GitHub.** Nếu cuộc thi chưa được chấp nhận điều khoản, hãy mở trang Instacart competition và chấp nhận rules trước.

In [14]:
from google.colab import files
KAGGLE_DIR = Path('/root/.kaggle')
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
if not KAGGLE_JSON.exists():
    print('Upload kaggle.json từ Kaggle Settings > API')
    up = files.upload()
    if 'kaggle.json' not in up:
        raise FileNotFoundError('Không thấy kaggle.json trong file được upload.')
    shutil.copy('/content/kaggle.json', KAGGLE_JSON)
    os.chmod(KAGGLE_JSON, 0o600)
print('Kaggle API token đã cấu hình (không in nội dung token).')

Kaggle API token đã cấu hình (không in nội dung token).


In [15]:
!pip -q install -U kagglehub

In [16]:
import kagglehub
from pathlib import Path

DATA_DIR = "/content/instacart"

download_path = kagglehub.dataset_download(
    "psparks/instacart-market-basket-analysis",
    output_dir=DATA_DIR
)

print("Dataset đã tải tại:", download_path)

Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
Dataset đã tải tại: /kaggle/input/instacart-market-basket-analysis


In [17]:
from pathlib import Path

root = Path(download_path)

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p.relative_to(root), f"({p.stat().st_size / 1e6:.1f} MB)")

aisles.csv (0.0 MB)
departments.csv (0.0 MB)
order_products__prior.csv (577.6 MB)
order_products__train.csv (24.7 MB)
orders.csv (109.0 MB)
products.csv (2.2 MB)


In [18]:
from pathlib import Path

root = Path(download_path)

orders_path = next(root.rglob("orders.csv"), None)

if orders_path is None:
    print("Không tìm thấy orders.csv tại:", root)
else:
    DATA_DIR = orders_path.parent

    print("Dataset directory:", DATA_DIR)

    required_files = [
        "orders.csv",
        "order_products__prior.csv",
        "order_products__train.csv",
        "products.csv",
        "aisles.csv",
        "departments.csv",
    ]

    for filename in required_files:
        path = DATA_DIR / filename
        status = "OK" if path.exists() else "MISSING"
        print(f"{status}: {filename}")

Dataset directory: /kaggle/input/instacart-market-basket-analysis
OK: orders.csv
OK: order_products__prior.csv
OK: order_products__train.csv
OK: products.csv
OK: aisles.csv
OK: departments.csv


In [19]:
import pandas as pd

for filename in [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
]:
    df = pd.read_csv(DATA_DIR / filename, nrows=5)

    print(f"\n--- {filename} ---")
    print("Columns:", df.columns.tolist())
    display(df)


--- orders.csv ---
Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0



--- order_products__prior.csv ---
Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0



--- order_products__train.csv ---
Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1



--- products.csv ---
Columns: ['product_id', 'product_name', 'aisle_id', 'department_id']


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## 3. Tải dữ liệu thẳng vào Colab

Ba file cần thiết cho thí nghiệm offline: `orders.csv`, `order_products__prior.csv`, `order_products__train.csv`. Phần nhãn của `eval_set=test` không được phát hành; ta dùng các order thuộc `eval_set=train` làm target có nhãn để đánh giá offline.

In [21]:
!pip -q install -U kagglehub

from pathlib import Path
import kagglehub

REQUIRED_FILES = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

def find_dataset_dir(search_root):
    search_root = Path(search_root)

    if not search_root.exists():
        return None

    for orders_file in search_root.rglob("orders.csv"):
        folder = orders_file.parent

        if all((folder / name).is_file() for name in REQUIRED_FILES):
            return folder

    return None


# Bước 1: Tìm dữ liệu đã tải trong Colab
DATA_DIR = find_dataset_dir("/content")

# Bước 2: Chỉ tải lại nếu không tìm thấy đủ 6 file
if DATA_DIR is None:
    output_dir = Path("/content/instacart")
    output_dir.mkdir(parents=True, exist_ok=True)

    download_path = kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis",
        output_dir=str(output_dir),
    )

    print("KaggleHub download path:", download_path)

    DATA_DIR = (
        find_dataset_dir(output_dir)
        or find_dataset_dir(download_path)
    )

if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ 6 file CSV. "
        "Hãy kiểm tra đường dẫn tải dữ liệu."
    )

print("\nDataset directory:", DATA_DIR)
print("\nKiểm tra file:")

for filename in REQUIRED_FILES:
    path = DATA_DIR / filename
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"OK | {filename} | {size_mb:.2f} MiB")

Using Colab cache for faster access to the 'instacart-market-basket-analysis' dataset.
KaggleHub download path: /kaggle/input/instacart-market-basket-analysis

Dataset directory: /kaggle/input/instacart-market-basket-analysis

Kiểm tra file:
OK | orders.csv | 103.92 MiB
OK | order_products__prior.csv | 550.80 MiB
OK | order_products__train.csv | 23.54 MiB
OK | products.csv | 2.07 MiB
OK | aisles.csv | 0.00 MiB
OK | departments.csv | 0.00 MiB


In [22]:
import pandas as pd

for filename in [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
]:
    df = pd.read_csv(DATA_DIR / filename, nrows=5)

    print(f"\n--- {filename} ---")
    print("Columns:", df.columns.tolist())
    display(df)


--- orders.csv ---
Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0



--- order_products__prior.csv ---
Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0



--- order_products__train.csv ---
Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1



--- products.csv ---
Columns: ['product_id', 'product_name', 'aisle_id', 'department_id']


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


## 4. Chạy kiểm tra schema

Đoạn này chỉ đọc header và vài dòng đầu, không tải nguyên 4 file CSV vào RAM.

In [23]:
import pandas as pd
for filename in required_files:
    path = DATA_DIR / filename
    header = pd.read_csv(path, nrows=0)
    sample = pd.read_csv(path, nrows=3)
    print('\n', filename, '| columns:', header.columns.tolist())
    display(sample)


 orders.csv | columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0



 order_products__prior.csv | columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0



 order_products__train.csv | columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0


## 5. Tạo dữ liệu thực nghiệm H = 4, 8, 13

Mặc định dùng 1.000 khách hàng đủ lịch sử để thử pipeline trước. Khi chạy thành công, tăng `MAX_USERS` hoặc đặt `0` để dùng tất cả user đủ điều kiện. `TOP_ITEMS=6000` giới hạn catalog trong lần chạy đầu; báo cáo sẽ ghi tỷ lệ sản phẩm target còn nằm trong catalog.

In [24]:
OUT_DIR = Path('/content/experiment_data')
HISTORY_LENGTHS = [4, 8, 13]
MAX_USERS = 1000       # smoke test; 0 = tất cả user đủ điều kiện
TOP_ITEMS = 6000
SEED = 42
CHUNK_SIZE = 500_000

cmd = [sys.executable, str(PROJECT_DIR / 'src' / 'prepare_instacart.py'),
       '--data_dir', str(DATA_DIR), '--out_dir', str(OUT_DIR),
       '--history_lengths', *map(str, HISTORY_LENGTHS),
       '--max_users', str(MAX_USERS), '--top_items', str(TOP_ITEMS),
       '--seed', str(SEED), '--chunk_size', str(CHUNK_SIZE)]
res = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(res.stdout[-20000:])
if res.returncode != 0:
    raise RuntimeError('Preprocessing thất bại. Xem traceback ở trên trước khi chạy model.')

summary = pd.read_csv(OUT_DIR / 'preparation_summary.csv')
display(summary)
print('Saved to:', OUT_DIR)

Loading order metadata...
Eligible labeled users selected: 1,000
Selected prior orders: 30,365
Read order_products__prior.csv: retained 309,935 matching item rows
Read order_products__train.csv: retained 10,912 matching item rows
H=4: wrote history/future files for 910 users
H=8: wrote history/future files for 910 users
H=13: wrote history/future files for 910 users
Final common cohort: 910 users
Candidate item count: 6,000
Target item coverage after candidate filtering: 87.28%
Outputs written to: /content/experiment_data
 H  users  train_users  valid_users  test_users  candidate_items  target_item_coverage                                       history_file                                       future_file
 4    910          677           68         165             6000              0.872815  /content/experiment_data/instacart_H4_history.csv  /content/experiment_data/instacart_H4_future.csv
 8    910          677           68         165             6000              0.872815  /content

,H,users,train_users,valid_users,test_users,candidate_items,target_item_coverage,history_file,future_file
0,4,910,677,68,165,6000,0.872815,/content/experiment_data/instacart_H4_history.csv,/content/experiment_data/instacart_H4_future.csv
1,8,910,677,68,165,6000,0.872815,/content/experiment_data/instacart_H8_history.csv,/content/experiment_data/instacart_H8_future.csv
2,13,910,677,68,165,6000,0.872815,/content/experiment_data/instacart_H13_history...,/content/experiment_data/instacart_H13_future.csv


Saved to: /content/experiment_data


## 6. Chạy baseline Popularity

Baseline này xếp hạng sản phẩm theo tần suất trong lịch sử và target đã gắn nhãn của nhóm train. Các metric được tính trên user test đã giữ riêng. Đây là mốc so sánh; không phải model chính.

In [25]:
cmd = [sys.executable, str(PROJECT_DIR / 'src' / 'evaluate_popularity.py'),
       '--prepared_dir', str(OUT_DIR),
       '--history_lengths', *map(str, HISTORY_LENGTHS),
       '--ks', '10', '20', '40']
res = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(res.stdout[-15000:])
if res.returncode != 0:
    raise RuntimeError('Đánh giá Popularity thất bại; kiểm tra log ở trên.')
baseline_results = pd.read_csv(OUT_DIR / 'popularity_baseline_metrics.csv')
display(baseline_results)

 H      Model  K  Test users  Precision@K  Recall@K   NDCG@K  HitRate@K
 4 Popularity 10         165     0.076364  0.078016 0.100960   0.496970
 4 Popularity 20         165     0.053333  0.111864 0.106945   0.581818
 4 Popularity 40         165     0.040152  0.162916 0.128315   0.654545
 8 Popularity 10         165     0.075152  0.078425 0.100698   0.496970
 8 Popularity 20         165     0.054848  0.115033 0.108443   0.593939
 8 Popularity 40         165     0.040606  0.163621 0.128576   0.642424
13 Popularity 10         165     0.075152  0.078425 0.100698   0.496970
13 Popularity 20         165     0.053939  0.114977 0.108037   0.587879
13 Popularity 40         165     0.041364  0.166790 0.129922   0.666667

Saved: /content/experiment_data/popularity_baseline_metrics.csv



,H,Model,K,Test users,Precision@K,Recall@K,NDCG@K,HitRate@K
0,4,Popularity,10,165,0.076364,0.078016,0.100960,0.496970
1,4,Popularity,20,165,0.053333,0.111864,0.106945,0.581818
2,4,Popularity,40,165,0.040152,0.162916,0.128315,0.654545
3,8,Popularity,10,165,0.075152,0.078425,0.100698,0.496970
4,8,Popularity,20,165,0.054848,0.115033,0.108443,0.593939
5,8,Popularity,40,165,0.040606,0.163621,0.128576,0.642424
6,13,Popularity,10,165,0.075152,0.078425,0.100698,0.496970
7,13,Popularity,20,165,0.053939,0.114977,0.108037,0.587879
8,13,Popularity,40,165,0.041364,0.166790,0.129922,0.666667


## 7. Lấy mã nguồn Sets2Sets của tác giả

Ta dùng mô hình đã công bố, không tự viết kiến trúc thay thế. Repository chính thức: [HaojiHu/Sets2Sets](https://github.com/HaojiHu/Sets2Sets). Source gốc ghi Python 3.6.8 / PyTorch 1.0.1 nên có thể không tương thích runtime Colab hiện tại.

In [26]:
REPO_DIR = Path('/content/Sets2Sets')
if not REPO_DIR.exists():
    clone = subprocess.run(['git', 'clone', 'https://github.com/HaojiHu/Sets2Sets.git', str(REPO_DIR)],
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(clone.stdout[-8000:])
    if clone.returncode != 0:
        raise RuntimeError('Không clone được source Sets2Sets. Kiểm tra kết nối mạng Colab.')
print('Repo exists:', REPO_DIR.exists())
print('Model file exists:', (REPO_DIR / 'Sets2Sets.py').exists())
print((REPO_DIR / 'README.md').read_text(encoding='utf-8')[:3000])

Cloning into '/content/Sets2Sets'...

Repo exists: True
Model file exists: True
# Sets2Sets

This is our implementation for the paper: 

Haoji Hu and Xiangnan He (2019). Sets2Sets: Learning from Sequential Sets with Neural Networks. In the 25th ACM SIGKDD Conference on Knowledge Discovery and Data Mining (KDD ’19), Anchorage, AK, USA

**Please cite our paper if you use our codes and datasets. Thanks!** 
```
@inproceedings{hu2019sets2sets,
  title={Sets2Sets: Learning from Sequential Sets with Neural Networks},
  author={Hu, Haoji and He, Xiangnan},
  booktitle={Proceedings of the 25th ACM SIGKDD international conference on Knowledge discovery and data mining},
  pages={1491--1499},
  year={2019},
  organization={ACM}
}
```

Author: Haoji Hu

## Environment Settings
We use pytorch to implement our method. 
- Torch version:  '1.0.1'
- Python version: '3.6.8'

## A quick start to run the codes with Ta-Feng data set.

Training:
```
python Sets2Sets.py ./data/TaFang_history.csv ./data/TaFan

In [29]:
from pathlib import Path

runner = Path(
    "/content/NextBasket_Instacart_Sets2Sets/src/run_sets2sets.py"
)

source = runner.read_text(encoding="utf-8")

old = '''    train_cmd = [sys.executable, str(script), str(history), str(future), model_name, str(args.num_future), "1"]
    test_cmd = [sys.executable, str(script), str(history), str(future), model_name, str(args.num_future), "0"]
'''

new = '''    # Sets2Sets.py prefixes input paths with "./".
    # Pass paths relative to the repository to avoid malformed absolute paths.
    import os
    history_arg = os.path.relpath(history, start=repo)
    future_arg = os.path.relpath(future, start=repo)

    train_cmd = [sys.executable, str(script), history_arg, future_arg,
                 model_name, str(args.num_future), "1"]
    test_cmd = [sys.executable, str(script), history_arg, future_arg,
                model_name, str(args.num_future), "0"]
'''

if old not in source:
    raise RuntimeError(
        "Không tìm thấy đoạn code gốc để thay thế. "
        "Dừng lại, không sửa tự động."
    )

runner.write_text(source.replace(old, new, 1), encoding="utf-8")
print("Đã sửa runner:", runner)

Đã sửa runner: /content/NextBasket_Instacart_Sets2Sets/src/run_sets2sets.py


## 8. Smoke test Sets2Sets với H=4

Chạy một H trước. Nếu lỗi, đoạn dưới sẽ in toàn bộ log cuối cùng và lưu log trong `experiment_data/logs/`. Không tiếp tục chạy H=8 và H=13 cho đến khi H=4 chạy thành công.

In [30]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(PROJECT_DIR / "src" / "run_sets2sets.py"),
    "--repo_dir", str(REPO_DIR),
    "--prepared_dir", str(OUT_DIR),
    "--h", "4",
    "--num_future", "1",
]

res = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(res.stdout[-30000:])

if res.returncode != 0:
    print("H=4 chưa chạy thành công.")
    print("Log:", OUT_DIR / "logs" / "sets2sets_H4_train.log")
else:
    print("H=4 train/test hoàn thành.")


$ /usr/bin/python3 /content/Sets2Sets/Sets2Sets.py ../experiment_data/instacart_H4_history.csv ../experiment_data/instacart_H4_future.csv Instacart_H4 1 1
start dictionary generation...
{'MATERIAL_NUMBER': 5502}
# dimensions of final vector: 5502 | 208
finish dictionary generation*****
num of vectors having entries more than 1: 3639
num of vectors having entries more than 1: 909
Number of training instances: 654
Traceback (most recent call last):
  File "/content/Sets2Sets/Sets2Sets.py", line 1036, in <module>
    main(sys.argv)
    ~~~~^^^^^^^^^^
  File "/content/Sets2Sets/Sets2Sets.py", line 983, in main
    trainIters(data_chunk, input_size, encoder1, attn_decoder1, model_version, training_key_set, weights,
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
               next_k_step, num_iter, print_every=print_val)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Sets2Sets/Sets2Sets.py", line 566,

## 9. Kết quả baseline và các file đầu ra

Các file được tạo trong `/content/experiment_data` gồm:

- `instacart_H4_history.csv`, `instacart_H4_future.csv` (tương tự H=8, H=13)
- `user_splits.csv`, `user_splits.json`
- `item_mapping.csv`
- `preparation_summary.csv`, `experiment_metadata.json`
- `popularity_baseline_metrics.csv`
- `logs/` nếu chạy Sets2Sets

**Cảnh báo học thuật:** Trước khi đối chiếu kết quả Sets2Sets với Popularity, cần xác minh source Sets2Sets dùng cùng cách chia 72/8/20 và cùng thứ tự user. Nếu không khớp, phải dùng một split chung hoặc benchmark implementation hỗ trợ keyset cố định.

In [ ]:
display(pd.read_csv(OUT_DIR / 'preparation_summary.csv'))
display(pd.read_csv(OUT_DIR / 'popularity_baseline_metrics.csv'))
print('Metadata:')
print((OUT_DIR / 'experiment_metadata.json').read_text(encoding='utf-8'))